In [2]:
import os
import pandas as pd

In [3]:
raw_dir = os.path.join("..", "data", "raw")
print(os.listdir(raw_dir))

['district_dogs_census_2012.csv', 'district_livestock_census.csv', 'Mortality districts_1989.csv', 'Mortality districts_1992.csv', 'Mortality districts_1995.csv', 'state_mortality_panel.dta', 'vulture_habitat.csv']


In [7]:
# combining three different mortality CSVs into one master mortality csv from 1988 to 1995
mortality89 = pd.read_csv(os.path.join(raw_dir, "Mortality districts_1989.csv"))
mortality92 = pd.read_csv(os.path.join(raw_dir, "Mortality districts_1992.csv"))
mortality95 = pd.read_csv(os.path.join(raw_dir, "Mortality districts_1995.csv"))

cols = ["state", "district", "year", "drural", "durban"]
m89 = mortality89[cols]
m92 = mortality92[cols]
m95 = mortality95[cols]

mortality = pd.concat([m89, m92, m95], ignore_index=True)

mortality.head()

,state,district,year,drural,durban
0,andhra pradesh,adilabad,1988,698,559
1,andhra pradesh,adilabad,1989,550,599
2,andhra pradesh,anantapur,1988,8331,2993
3,andhra pradesh,anantapur,1989,6884,3146
4,andhra pradesh,chittoor,1988,7481,3074


In [23]:
#summarizing mortality stats
mortality["total_deaths"] = mortality["drural"].fillna(0) + mortality["durban"].fillna(0)

print("Summary statistics for total deaths:")
print(mortality["total_deaths"].describe())
mortality.head()

Summary statistics for total deaths:
count     3741.000000
mean      7204.727346
std       8118.202849
min          0.000000
25%       1845.000000
50%       5323.000000
75%       9826.000000
max      80104.000000
Name: total_deaths, dtype: float64


,state,district,year,drural,durban,total_deaths
0,andhra pradesh,adilabad,1988,698.0,559.0,1257.0
1,andhra pradesh,adilabad,1989,550.0,599.0,1149.0
2,andhra pradesh,anantapur,1988,8331.0,2993.0,11324.0
3,andhra pradesh,anantapur,1989,6884.0,3146.0,10030.0
4,andhra pradesh,chittoor,1988,7481.0,3074.0,10555.0


In [29]:
# vulture habitat dataset
vulture_raw = pd.read_csv(os.path.join(raw_dir, "vulture_habitat.csv"))

# binary habitat indicator: 1 if district had Gyps indicus vultures, else 0 ) (We only choose Gyps Indicus as our species of focus as we have enough data for disctricts where it was absent and present 272(present) vs 70(absent) , also stated in paper)
vulture_raw["vulture_habitat"] = (vulture_raw["Gyps indicus0"] == 1).astype(int)

# Keep only the identifier columns and the final binary habitat flag
vulture_df = vulture_raw[["state", "district", "vulture_habitat"]].copy()

print("Vulture habitat distribution across districts:")
print(vulture_df["vulture_habitat"].value_counts())
vulture_df.head()

Vulture habitat distribution across districts:
vulture_habitat
1    272
0     70
Name: count, dtype: int64


,state,district,vulture_habitat
0,andaman & nicobar islands,andaman & nicobar islands,0
1,andhra pradesh,adilabad,1
2,andhra pradesh,anantapur,1
3,andhra pradesh,chittoor,1
4,andhra pradesh,cuddapah,1


In [42]:
# livestock census dataset
livestock_raw = pd.read_csv(os.path.join(raw_dir, "district_livestock_census.csv"))

livestock_df = livestock_raw[["state", "district", "year", "cattle", "dog_govt"]].copy()

#common lowercase naming
livestock_df["state"] = livestock_df["state"].astype(str).str.strip().str.lower()
livestock_df["district"] = livestock_df["district"].astype(str).str.strip().str.lower()

# before collapse years 1987 to 1993
before_collapse = livestock_df["year"].isin([1987, 1988, 1990, 1992, 1993])
livestock_before = livestock_df[before_collapse]

# avg number per district before collapse
livestock_before_collapse = livestock_before.groupby(["state", "district"])[["cattle", "dog_govt"]].mean().reset_index()

print("Districts with baseline livestock:", livestock_before_collapse.shape[0])
livestock_before_collapse.head()

Districts with baseline livestock: 386


,state,district,cattle,dog_govt
0,andhra pradesh,ananthapur,612.605,26.9820
1,andhra pradesh,chittoor,829.720,36.7020
2,andhra pradesh,east godavari,371.760,32.1100
3,andhra pradesh,guntur,265.100,10.3930
4,andhra pradesh,kadapa,NaN,67.5075


In [43]:
# merge mortality data with the vulture habitat indicator on state and district
df_merged = pd.merge(mortality, vulture_df, on=["state", "district"], how="inner")

# merge in the baseline livestock and dog counts
df_final = pd.merge(df_merged, livestock_before_collapse, on=["state", "district"], how="inner")

# save the clean analysis dataset to data/clean/
output_path = os.path.join("..", "data", "clean", "district_analysis_panel.csv")
df_final.to_csv(output_path, index=False)

print("Final merged dataset shape:", df_final.shape)
print("Unique districts:", df_final["district"].nunique())
print("Years included:", sorted(df_final["year"].unique()))
df_final.head()

Final merged dataset shape: (1400, 9)
Unique districts: 181
Years included: [np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995)]


,state,district,year,drural,durban,total_deaths,vulture_habitat,cattle,dog_govt
0,andhra pradesh,chittoor,1988,7481.0,3074.0,10555.0,1,829.72,36.702
1,andhra pradesh,chittoor,1989,7279.0,3143.0,10422.0,1,829.72,36.702
2,andhra pradesh,east godavari,1988,5009.0,5935.0,10944.0,1,371.76,32.110
3,andhra pradesh,east godavari,1989,3782.0,6103.0,9885.0,1,371.76,32.110
4,andhra pradesh,guntur,1988,9068.0,6322.0,15390.0,1,265.10,10.393
